# Land-use Mapping Audit

This notebook summarizes the harmonized land-use taxonomy, mapping audit, and city-level class counts used by the CityRep land-use task.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists():
    ROOT = ROOT.parent.resolve()

RESULTS = ROOT / 'results'
MAPPING_RESULTS = RESULTS / 'mapping'
FIG_DIR = MAPPING_RESULTS / 'figures'
DATA_RELEASE = ROOT / 'data'
FIG_DIR.mkdir(parents=True, exist_ok=True)

mapping_path = MAPPING_RESULTS / 'landuse_mapping_table_readable.csv'
audit_path = MAPPING_RESULTS / 'landuse_mapping_audit.csv'
mapping = pd.read_csv(mapping_path).fillna('')
audit = pd.read_csv(audit_path).fillna('')

CITY_ORDER = ['new_york', 'singapore', 'sydney', 'mumbai', 'nairobi', 'jakarta', 'cape_town']
CITY_LABEL = {
    'new_york':'New York', 'singapore':'Singapore', 'sydney':'Sydney',
    'mumbai':'Mumbai', 'nairobi':'Nairobi', 'jakarta':'Jakarta', 'cape_town':'Cape Town',
}
mapping = mapping[mapping['city'].isin(CITY_ORDER)].copy()
audit = audit[audit['city'].isin(CITY_ORDER)].copy()

CLASS_ORDER = [
    'Residential', 'Mixed Use', 'Commercial', 'Industrial', 'Transportation',
    'Green / Recreation', 'Institutional / Civic', 'Utilities', 'Water',
    'Agriculture / Rural', 'Vacant / Reserve', 'Other',
]

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 120)
sns.set_theme(style='whitegrid', context='notebook')
print('Root:', ROOT)
print('Mapping table:', mapping_path)
print('Audit table:', audit_path)


## Common Taxonomy

All city-specific land-use labels are mapped to the same 12-class taxonomy before sampling the benchmark point labels.


In [ ]:
taxonomy = pd.DataFrame({'label_id': range(len(CLASS_ORDER)), 'mapped_group': CLASS_ORDER})
taxonomy.to_csv(MAPPING_RESULTS / 'landuse_common_taxonomy.csv', index=False)
display(taxonomy)


## Mapping Audit Summary

The table below records the basic audit status of the released mapping files. The full readable mapping table is kept in `results/mapping/landuse_mapping_table_readable.csv`.


In [ ]:
summary_rows = {
    'mapping_rows': len(mapping),
    'audit_rows': len(audit),
    'cities': ', '.join(CITY_LABEL[c] for c in CITY_ORDER),
    'mapped_groups': len(CLASS_ORDER),
    'empty_mapped_group_rows': int((mapping['mapped_group'].astype(str).str.len() == 0).sum()),
}
if 'explicit_mapping' in audit.columns:
    explicit = audit['explicit_mapping'].astype(str).str.lower().isin(['true', '1', 'yes'])
    summary_rows['explicit_mapping_rows'] = int(explicit.sum())
    summary_rows['non_explicit_mapping_rows'] = int((~explicit).sum())

summary_table = pd.DataFrame(summary_rows.items(), columns=['item', 'value'])
summary_table.to_csv(MAPPING_RESULTS / 'landuse_mapping_audit_summary.csv', index=False)
display(summary_table)

city_mapping_counts = (
    mapping.groupby(['city', 'mapped_group'])
    .size()
    .reset_index(name='n_source_rows')
)
city_mapping_counts['city_label'] = city_mapping_counts['city'].map(CITY_LABEL)
city_mapping_counts.to_csv(MAPPING_RESULTS / 'landuse_mapping_rows_by_city_class.csv', index=False)
display(
    city_mapping_counts.pivot(index='city_label', columns='mapped_group', values='n_source_rows')
    .reindex([CITY_LABEL[c] for c in CITY_ORDER])[CLASS_ORDER]
    .fillna(0).astype(int)
)


## Land-use Class Counts by City

Each released land-use task contains 100,000 point samples per city. The figure shows the number of samples assigned to each harmonized land-use class.


In [ ]:
rows = []
for city in CITY_ORDER:
    task_dir = DATA_RELEASE / 'tasks' / f'{city}.landuse.2026'
    sample_path = task_dir / 'samples.parquet'
    task_json = json.loads((task_dir / 'task.json').read_text())
    df = pd.read_parquet(sample_path, columns=['label'])
    counts = df['label'].value_counts().reindex(CLASS_ORDER, fill_value=0)
    total = int(counts.sum())
    for cls in CLASS_ORDER:
        rows.append({
            'city': city,
            'city_label': CITY_LABEL[city],
            'mapped_group': cls,
            'count': int(counts[cls]),
            'percent': float(counts[cls] / total * 100) if total else 0.0,
            'task_n_samples': int(task_json.get('n_samples', total)),
        })

class_counts = pd.DataFrame(rows)
class_counts.to_csv(MAPPING_RESULTS / 'landuse_class_counts_by_city.csv', index=False)

count_table = class_counts.pivot(index='city_label', columns='mapped_group', values='count').reindex(
    [CITY_LABEL[c] for c in CITY_ORDER]
)[CLASS_ORDER].fillna(0).astype(int)
display(count_table)

palette = sns.color_palette('tab20', n_colors=len(CLASS_ORDER))
fig, ax = plt.subplots(figsize=(13.5, 5.6))
count_table.plot(kind='bar', stacked=True, color=palette, width=0.82, ax=ax)
ax.set_title('Land-use class counts by city')
ax.set_xlabel('')
ax.set_ylabel('Number of samples')
ax.tick_params(axis='x', rotation=25)
ax.legend(title='Class', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
fig.tight_layout()
out = FIG_DIR / 'landuse_class_counts_by_city.png'
fig.savefig(out, dpi=260, bbox_inches='tight')
print(out)
plt.show()


## Paper-ready Outputs

This notebook writes the taxonomy, audit summary, class-count table, and class-count figure under `results/mapping/`.


In [ ]:
outputs = [
    MAPPING_RESULTS / 'landuse_common_taxonomy.csv',
    MAPPING_RESULTS / 'landuse_mapping_audit_summary.csv',
    MAPPING_RESULTS / 'landuse_mapping_rows_by_city_class.csv',
    MAPPING_RESULTS / 'landuse_class_counts_by_city.csv',
    FIG_DIR / 'landuse_class_counts_by_city.png',
]
for out in outputs:
    print(out)
